In [ ]:
import sys
import os
import datasets
from typing import List, Dict, Any
import random
import json

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils.dataset_tokenization import extract_deleted_text
from src.utils.formatting import apply_del_w_tokens

In [19]:
def format_single_correct_sample(sample: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Formats a single raw sample into one or more training samples."""

    sample_kwargs = {
        "question": sample["question"],
        "answer": str(sample.get("answer", [""])[0]),
    }
    if "is_answerable" in sample:
        sample_kwargs["is_answerable"] = sample["is_answerable"]
    else:
        sample_kwargs["context"] = sample.get("context", "")
    
    formatted_samples = []
    responses = sample["responses"]
    # sort responses by length
    responses.sort(key=len)
    # samples_to_include = 2 if sample["is_answerable"] else 1
    samples_to_include = 1


    for i in range(samples_to_include):
        formatted_samples.append({
            "input": sample["input"],
            "incorrect_response": "",
            "errors": [],
            "hallucinated_text": [],
            "correct_response": responses[-i],
            "additional_info": sample_kwargs,
        })

    return formatted_samples

In [ ]:
def format_single_incorrect_sample(sample: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Formats a single raw sample into one or more training samples."""
    if sample.get("wrong_response_number", 0) == 0:
        return []

    for correct_response in sample["corrected_responses"]:
        if ("<DEL_S>" not in correct_response and "<DEL_A>" not in correct_response):
            return []

    sample_kwargs = {
        "question": sample["question"],
        "answer": str(sample.get("answer", [""])[0]),
    }
    if "is_answerable" in sample:
        sample_kwargs["is_answerable"] = sample["is_answerable"]
    else:
        sample_kwargs["context"] = sample.get("context", "")
    
    formatted_samples = []
    correct_response_index = 0

    # samples_to_include = 2 if random.random() < 0.35 else 3
    samples_to_include = 1

    for i, is_verified in enumerate(sample.get("verified_response_mask", [])):
        if is_verified:
            hallucinated_text = []
            for error in sample["errors_to_correct"][i]:
                deleted_text = extract_deleted_text(error["correction"])
                if deleted_text:
                    hallucinated_text.append(deleted_text)

            if "<DEL_W>" in sample["corrected_responses"][correct_response_index]:
                sample["corrected_responses"][correct_response_index] = apply_del_w_tokens(sample["corrected_responses"][correct_response_index])

            if hallucinated_text:
                formatted_samples.append({
                    "input": sample["input"],
                    "incorrect_response": sample["responses_to_correct"][i],
                    "errors": sample["errors_to_correct"][i],
                    "hallucinated_text": hallucinated_text,
                    "correct_response": sample["corrected_responses"][correct_response_index],
                    "additional_info": sample_kwargs,
                })
                correct_response_index += 1
        
        if correct_response_index >= samples_to_include:
            break

    return formatted_samples

In [27]:
context_data_path = "../../dataset/processed_data/train/rajpurkar_squad_processed.json"
context_dataset = datasets.load_dataset("json", data_files=context_data_path)

math_data_path = "../../dataset/processed_data/train/UMWP_processed.json"
math_dataset = datasets.load_dataset("json", data_files=math_data_path)

In [28]:
print(context_dataset)
print(math_dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'question', 'context', 'answer', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
        num_rows: 87557
    })
})
DatasetDict({
    train: Dataset({
        features: ['input', 'question', 'answer', 'is_answerable', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
        num_rows: 4412
    })
})


## Math QA dataset prep

In [29]:
final_math_dataset = []

### Find incorrect samples that are answerable

In [30]:
incorrect_answerable_samples = math_dataset.filter(lambda x: x["wrong_response_number"] > 0 and x["is_answerable"])
incorrect_answerable_samples = incorrect_answerable_samples["train"]
print(incorrect_answerable_samples)

Dataset({
    features: ['input', 'question', 'answer', 'is_answerable', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
    num_rows: 727
})


In [31]:
incorrect_answerable_samples[2]

{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a meticulous AI mathematician. Your task is to solve the following math problem.\n\nFollow these steps carefully:\n1. **Analyze the problem:** First, understand the given information and what is being asked.\n2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.\n3. **Solve or Explain:**\n   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.\n   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.\n\nYour entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nAfter eating a hearty meal 

In [32]:
for sample in incorrect_answerable_samples:
    final_math_dataset.extend(format_single_incorrect_sample(sample))

random.shuffle(final_math_dataset)
print(len(final_math_dataset))

895


In [33]:
final_math_dataset[0]

{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a meticulous AI mathematician. Your task is to solve the following math problem.\n\nFollow these steps carefully:\n1. **Analyze the problem:** First, understand the given information and what is being asked.\n2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.\n3. **Solve or Explain:**\n   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.\n   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.\n\nYour entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nJames has 7 more than 4 tim

### Find incorrect samples that are unanswerable

In [35]:
incorrect_unanswerable_samples = math_dataset.filter(lambda x: x["wrong_response_number"] > 0 and not x["is_answerable"])
incorrect_unanswerable_samples = incorrect_unanswerable_samples["train"]
print(incorrect_unanswerable_samples)

Dataset({
    features: ['input', 'question', 'answer', 'is_answerable', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
    num_rows: 2110
})


In [36]:
incorrect_unanswerable_samples[0]

{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a meticulous AI mathematician. Your task is to solve the following math problem.\n\nFollow these steps carefully:\n1. **Analyze the problem:** First, understand the given information and what is being asked.\n2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.\n3. **Solve or Explain:**\n   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.\n   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.\n\nYour entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nMikaela was repainting her 

In [37]:
for i in range(2000):
    final_math_dataset.extend(format_single_incorrect_sample(incorrect_unanswerable_samples[i]))

random.shuffle(final_math_dataset)
print(len(final_math_dataset))

2892


### Find correct samples

In [40]:
# find data samples for math dataset with no errors 
correct_samples = math_dataset.filter(lambda x: x["wrong_response_number"] == 0)
correct_samples = correct_samples["train"]
print(correct_samples)

Dataset({
    features: ['input', 'question', 'answer', 'is_answerable', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
    num_rows: 1575
})


In [ ]:
for i in range(1100):
    final_math_dataset.extend(format_single_correct_sample(correct_samples[i]))

random.shuffle(final_math_dataset)
print(len(final_math_dataset))

5026


In [42]:
final_math_dataset[0]

{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a meticulous AI mathematician. Your task is to solve the following math problem.\n\nFollow these steps carefully:\n1. **Analyze the problem:** First, understand the given information and what is being asked.\n2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.\n3. **Solve or Explain:**\n   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.\n   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.\n\nYour entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nSam is twice as old as Sue.

In [43]:
output_path = os.path.join(project_root, "math_qa_train_dataset_s1.json")
with open(output_path, "w") as f:
    json.dump(final_math_dataset, f, indent=4)

## Context QA dataset

In [21]:
final_context_dataset = []

### Find incorrect samples

In [22]:
# find data samples for context dataset with no errors 
incorrect_samples = context_dataset.filter(lambda x: x["wrong_response_number"] > 0)
incorrect_samples = incorrect_samples["train"]
print(incorrect_samples)

Dataset({
    features: ['input', 'question', 'context', 'answer', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
    num_rows: 17354
})


In [ ]:
def format_single_incorrect_sample(sample: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Formats a single raw sample into one or more training samples."""
    if sample.get("wrong_response_number", 0) == 0:
        return []

    for correct_response in sample["corrected_responses"]:
        if "<DEL_W>" in correct_response:
            return []

    answer = sample["answer"][0].lower()
    counter = 0
    for response in sample["responses_to_correct"]:
        if answer in response.lower():
            counter += 1
        if counter > 2:
            return []

    sample_kwargs = {
        "question": sample["question"],
        "answer": str(sample.get("answer", [""])[0]),
    }
    if "is_answerable" in sample:
        sample_kwargs["is_answerable"] = sample["is_answerable"]
    else:
        sample_kwargs["context"] = sample.get("context", "")
    
    formatted_samples = []
    correct_response_index = 0

    # samples_to_include = 1 if random.random() < 0.05 else 2
    samples_to_include = 2

    for i, is_verified in enumerate(sample.get("verified_response_mask", [])):
        if is_verified:
            hallucinated_text = []
            for error in sample["errors_to_correct"][i]:
                deleted_text = extract_deleted_text(error["correction"])
                if deleted_text:
                    hallucinated_text.append(deleted_text)

            if hallucinated_text:
                formatted_samples.append({
                    "input": sample["input"],
                    "incorrect_response": sample["responses_to_correct"][i],
                    "errors": sample["errors_to_correct"][i],
                    "hallucinated_text": hallucinated_text,
                    "correct_response": sample["corrected_responses"][correct_response_index],
                    "additional_info": sample_kwargs,
                })
                correct_response_index += 1
        
        if correct_response_index >= samples_to_include:
            break

    return formatted_samples

In [33]:
final_context_dataset = []
samples_to_include = 12500
for i in range(samples_to_include):
    final_context_dataset.extend(format_single_incorrect_sample(incorrect_samples[i]))

random.shuffle(final_context_dataset)
print(len(final_context_dataset))

9024


In [34]:
final_context_dataset[0]

{'input': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a specialized question-answering AI. Your task is to give a concise answer to the question using *only* the provided context. Make sure to always give an answer.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContext:\n\'\'\'\nHistorians agree that Napoleon\'s remarkable personality was one key to his influence. They emphasize the strength of his ambition that took him from an obscure village to command of most of Europe. George F. E. Rudé stresses his "rare combination of will, intellect and physical vigour." At 5 ft 6 in (168 cm), he was not physically imposing but in one-on-one situations he typically had a hypnotic impact on people and seemingly bent the strongest leaders to his will. He understood military technology, but was not an innovator in that regard. He was an innovator in using the financial, bureaucratic, and diplomatic resources of France. He could rapidly dictate a series of complex

### Find correct samples

In [35]:
# find data samples for context dataset with no errors 
correct_samples = context_dataset.filter(lambda x: x["wrong_response_number"] == 0)
correct_samples = correct_samples["train"]
print(correct_samples)

Dataset({
    features: ['input', 'question', 'context', 'answer', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
    num_rows: 70203
})


In [36]:
samples_to_include = 6000
for i in range(samples_to_include):
    final_context_dataset.extend(format_single_correct_sample(correct_samples[i]))

random.shuffle(final_context_dataset)
print(len(final_context_dataset))

15024


In [37]:
output_path = os.path.join(project_root, "context_qa_train_dataset_s1.json")
with open(output_path, "w") as f:
    json.dump(final_context_dataset, f, indent=4)

### Dataset fusion

In [38]:
# load 2 datasets and fuse them
math_dataset = datasets.load_dataset("json", data_files=f"{project_root}/math_qa_train_dataset_s1.json")
context_dataset = datasets.load_dataset("json", data_files=f"{project_root}/context_qa_train_dataset_s1.json")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [39]:
final_dataset = []
math_dataset = math_dataset["train"].to_list()
context_dataset = context_dataset["train"].to_list()

final_dataset.extend(math_dataset)
final_dataset.extend(context_dataset)
random.shuffle(final_dataset)
print(len(final_dataset))

20050


In [40]:
output_path = os.path.join(project_root, "final_train_dataset_s1.json")
with open(output_path, "w") as f:
    json.dump(final_dataset, f, indent=4)

In [ ]:
import datasets

import sys
import os
# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [30]:
data_path = "../../dataset/final_train_dataset.json"
dataset = datasets.load_dataset("json", data_files=data_path)
dataset = dataset["train"].to_list()

In [31]:
len(dataset)

35022